# Exploratory synchronized sample clustering

Requires acquisition timezone metadata and MMS_CLOCKS_SYNCHRONIZED=1 from documented synchronization. Channels and visits remain separate. Backward eye match <=100ms, valid pupil, confidence-filtered HR and interval bounds 300–2000ms. K-means uses standardized features; PCA only displays them. Clusters are sample groups, not participant or clinical classes.

Data: `MMS_DATA_ROOT` or the checkout dataset. Exports: optional separate `MMS_OUTPUT_ROOT`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mms import exploration as ex
SOURCE = 'case-study'
aligned = [ex.synchronized_features(s, source=SOURCE) for s in (1, 2, 3)]
nonempty = [frame.assign(Session=s) for s, frame in enumerate(aligned, 1) if not frame.empty]
if nonempty:
    for session, frame in pd.concat(nonempty).groupby("Session"):
        for channel, part in frame.groupby("Channel"):
            features = part[["ibi", "heart_rate", "pupil"]]
            max_k = min(10, len(features.dropna()) - 1, len(features.dropna().drop_duplicates()) - 1)
            scores = []
            for k in range(2, max_k + 1):
                projection, score = ex.cluster_features(features, n_clusters=k)
                scores.append((k, score))
            if scores:
                best_k = max(scores, key=lambda pair: pair[1])[0]
                projection, score = ex.cluster_features(features, n_clusters=best_k)
                print(f"Session {session}, channel {channel}: in-sample k={best_k}, silhouette={score:.3f}")
                plt.scatter(projection.PC1, projection.PC2, c=projection.Cluster, s=8)
                plt.title("Exploratory sample clusters; PCA display only")
                plt.xlabel("PC1")
                plt.ylabel("PC2")
                plt.show()
                plt.close()
            else:
                print("Clustering skipped: insufficient distinct complete rows.")
